In [1]:
using ScenTrees, JLD2
include("functions.jl")

In [8]:
Case = "CCD22-3"

POIs=["WCASCADE", "JohnDay", "COTWDPGE", "Tesla", "Mossland"]
POI = POIs[1]

# cable length in meter
CabL = Dict("WCASCADE"=>545.060*1e3, "JohnDay"=>444.423*1e3, "COTWDPGE"=>324.193*1e3, "Tesla"=>603.598*1e3, "Mossland"=>660.732*1e3);

SolLoc = Case*"/"

KernTree = load(Case*"/ScenTree_"*POI*".jld2", "KernTree")

CabLength = CabL[POI];

N1 = 24; N2 = 4
T1 = [1:N1;]; # set of 24 hours in a day
T2 = [1:N1*N2;]; # set of 96 quarters in a day
T21 = Vector{Int64}[]; # set of 4 quarters in each hour
for t1 in T1
    push!(T21, [(t1-1)*N2+1:t1*N2;])
end

S1 = load(SolLoc*POI*"_Sol.jld2", "S1") # DA stage scenarios
S2 = load(SolLoc*POI*"_Sol.jld2", "S2") # RT stage scenarios
S21 = load(SolLoc*POI*"_Sol.jld2", "S21") # RT stage scenarios follow each DA stage scenario
pWS = load(SolLoc*POI*"_Sol.jld2", "pWS") # wind farm actual output T1, S2
pWDS = load(SolLoc*POI*"_Sol.jld2", "pWDS") # power traded in DA market T1, S1
pWRS = load(SolLoc*POI*"_Sol.jld2", "pWRS") # power traded in RT market T2, S2
pRWUS = load(SolLoc*POI*"_Sol.jld2", "pRWUS") # up reserve from wind farm T2, S2
pRWDS = load(SolLoc*POI*"_Sol.jld2", "pRWDS") # down reserve from wind farm T2, S2
pRBUS = load(SolLoc*POI*"_Sol.jld2", "pRBUS") # up reserve from battery T2, S2
pRBDS = load(SolLoc*POI*"_Sol.jld2", "pRBDS") # down reserve from battery T2, S2
ChS = load(SolLoc*POI*"_Sol.jld2", "ChS") # battery charged power T2, S2
DisS = load(SolLoc*POI*"_Sol.jld2", "DisS") # battery discharged power T2, S2
SCS = load(SolLoc*POI*"_Sol.jld2", "SCS") # battery state of charge
kWS = load(SolLoc*POI*"_Sol.jld2", "kWS") # control parameter for reserve from wind farm T2, S2
kBS = load(SolLoc*POI*"_Sol.jld2", "kBS") # control parameter for reserve from battery T2, S2
lam_DAQ = load(SolLoc*POI*"_Sol.jld2", "lam_DAQ") # DA price interpolated to T2
# DA price should be in T1 (hourly). I interpolated into T2 (every 15 minutes) for visualization
pWSQ = load(SolLoc*POI*"_Sol.jld2", "pWSQ") # wind farm actual output interpolated to T2
pWDSQ = load(SolLoc*POI*"_Sol.jld2", "pWDSQ") # power traded in DA market interpolated to T2
WPQ = load(SolLoc*POI*"_Sol.jld2", "WPQ") # wind farm generation capacity interpolated to T2
# results of passing wind speed to wind farm power curve
WSQ = load(SolLoc*POI*"_Sol.jld2", "WSQ") # wind speed interpolated to T2
lam_RT = load(SolLoc*POI*"_Sol.jld2", "lam_RT") # RT price T2, S2
PowerBase = load(SolLoc*POI*"_Sol.jld2", "PowerBase") # power base
Wrate = load(SolLoc*POI*"_Sol.jld2", "Wrate") # wind farm rated power

P1 = Dict() # probability of each DA stage scenario
P2 = Dict() # probability of each RT stage scenario
P21 = Dict() # probability of RT stage sceanrios for given DA stage scenario
for s in S1
    P1[s] = KernTree.probability[s]
end
for s in S2
    P2[s] = KernTree.probability[s]*KernTree.probability[KernTree.parent[s]]
    P21[s] = KernTree.probability[s]
end

States = Interpolate_States(KernTree);

lam_DA = Dict()
lam_RT = Dict()
lam_ReU = Dict()
lam_ReD = Dict()
WS = Dict()

for s in S1
    lam_DA[s] = KernTree.state[s, :] # DA price scenarios
end
for s in S2
    WS[s] = KernTree.state[s, :]
    col = KernTree.children[s+1][1]
    lam_RT[s] = States[col, :] # RT price scenarios
    col = KernTree.children[col+1][1]
    lam_ReU[s] = States[col, :] # Up reserve price scenarios
    col = KernTree.children[col+1][1]
    lam_ReD[s] = States[col, :] # Down reserve price scenarios
end
    
CabC = 2*0.3476*0.3351*1e6/525/1e3*CabLength # Cable material cost $/MW
CabFixC = 2*0.3476*0.0001*CabLength # Cable fixed cost
CabIntC = 2.7*1e6/16000*CabLength; # Cable installation cost
lam_ESS = 1.27009e6 # BESS cost $/MW, 4h duration

# other fixed cost
OnConv = 450*1e6 # onshore converter
OffConv = 772*1e6; # floating offshore converter

In [9]:
DCFR = sum(1/(1+0.03)^i for i=0:15) # discount cash flow rate
DAR = sum(lam_DA[s][t]*pWDS[s][t]*P1[s] for t in T1, s in S1)/1e3 # DA revenue $k
RTR = sum(lam_RT[s][t]*pWRS[s][t]*P2[s] for t in T2, s in S2)/4/1e3 # RT revenue $k
REUR = sum(lam_ReU[s][t]*(pRWUS[s][t]+pRBUS[s][t])*P2[s] for t in T2, s in S2)/4/1e3 # up reserve revenue $k
REDR = sum(lam_ReD[s][t]*(pRWDS[s][t]+pRBDS[s][t])*P2[s] for t in T2, s in S2)/4/1e3 # down reserve revenue $k
LR = DCFR*365*(DAR+RTR+REUR+REDR)/1e3 # life time revenue $M
;